### Imports and global setup

In [1]:
## conda env: stereo_visionn
import os
import cv2
import numpy as np
from rtmlib import Wholebody, draw_skeleton
from Util.util import BodyWithFeet, PoseTracker, Body

In [2]:
device = "cuda"  # cpu, cuda, mps
backend = "onnxruntime"  # opencv, onnxruntime, openvino
openpose_skeleton = False  # True for openpose-style, False for mmpose-style

body = Body(
    to_openpose=openpose_skeleton, mode="performance", backend=backend, device=device
)

load C:\Users\unger\.cache\rtmlib\hub\checkpoints\yolox_x_8xb8-300e_humanart-a39d44ed.onnx with onnxruntime backend
load C:\Users\unger\.cache\rtmlib\hub\checkpoints\rtmpose-x_simcc-body7_pt-body7_700e-384x288-71d7b7e9_20230629.onnx with onnxruntime backend


### Mapping keypoints

In [3]:
img = cv2.imread("./Util/tennis.webp", cv2.IMREAD_COLOR)

keypoints, scores = body(img)
img_show = np.zeros(img.shape, dtype=np.uint8)
img_show = draw_skeleton(img, keypoints, scores, kpt_thr=0.5)

cv2.imshow(
    "Image",
    cv2.resize(
        img_show, (round(img_show.shape[1] * 0.5), round(img_show.shape[0] * 0.5))
    ),
)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [6]:
img = cv2.imread("./Util/tennis.webp", cv2.IMREAD_COLOR)
for index, point in enumerate(keypoints[0]):
    cv2.circle(img, (round(point[0]), round(point[1])), 5, (0, 0, 255), 3)
    cv2.putText(img,f"{index}",(round(point[0] + 5), round(point[1]) + 5),cv2.FONT_HERSHEY_SIMPLEX,1.5,(255, 100, 0),3)
    print(point, index)

cv2.imshow(
    "Image", cv2.resize(img, (round(img.shape[1] * 0.5), round(img.shape[0] * 0.5)))
)
cv2.imwrite("./Util/Body_marker_locations.png", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

[1165.26229477  217.84829187] 0
[1154.22600047  206.81199789] 1
[1126.63526471  201.2938509 ] 2
[1126.63526471  239.92087984] 3
[1043.86305745  234.40273285] 4
[1060.41749891  328.21123171] 5
[1076.97194036  355.80196667] 6
[1209.40747197  449.61046553] 7
[1248.03450203  504.79193544] 8
[1214.92561913  388.91084862] 9
[1292.17967923  438.57417154] 10
[961.09085019 736.5541091 ] 11
[861.76420148 731.0359621 ] 12
[1187.33488337  907.61666584] 13
[663.11090406 979.35257673] 14
[1341.84300359 1172.48772144] 15
[359.61281077 940.72554779] 16


### Video analysis

In [6]:
keypoints_over_time = []

input_folder = "./stereo_videos"
file_name = "50cm_walk_1_SVGA120FPS_low_pos.avi"
cap = cv2.VideoCapture(os.path.join(input_folder, file_name))

if cap.isOpened() == False:
    print("Error opening video file")

# Read until video is completed
while cap.isOpened():

    # Capture frame-by-frame
    ret, frame = cap.read()

    if ret == True and frame is not None:
        width = frame.shape[1]
        height = frame.shape[0]

        # keypoints, scores = wholebody(frame)
        # keypoints, scores = halpe26(frame)
        # keypoints, scores = pose_tracker(frame)
        keypoints, scores = body(frame)

        ## clipping keypoint coordinates so they dont exceed image boundary (e.g., (-100, 5000))
        clipped_keypoints = np.transpose(np.array([np.clip(keypoints[0,:,0],0,1920),np.clip(keypoints[0,:,1],0,1200)]))


        keypoints_over_time.append(clipped_keypoints)

        ## if you want to use black background instead of original image,
        # img_show = np.zeros(frame.shape, dtype=np.uint8)
        # img_show = draw_skeleton(frame, keypoints, scores, kpt_thr=0.6)

        img_show = draw_skeleton(frame, np.array([clipped_keypoints]), scores, kpt_thr=0.6)

        ## check to see in ankle tracking switches when changing direction
        # cv2.circle(img_show,(round(keypoints[0][16][0]),round(keypoints[0][16][1])),5,(0,0,255),3 )

        cv2.imshow("Image", cv2.resize(img_show, (800, 600)))

        # Press Q on keyboard to exit
        if cv2.waitKey(25) & 0xFF == ord("q"):
            break

    else:
        break


cap.release()
cv2.destroyAllWindows()

keypoints_over_time = np.array(keypoints_over_time)
print(keypoints_over_time.shape)

(4871, 17, 2)


In [7]:
keypoints_over_time.shape

(4871, 17, 2)

In [9]:
for points in keypoints_over_time:
    print(points[15:17])
    break

[[285.   650.  ]
 [316.25 643.75]]


#### Save the coords of chosen keypoints

In [8]:
out_folder_name = "./saved_coords"
np.save(
    f"{os.path.join(out_folder_name, file_name)}_body06_performance.npy", keypoints_over_time
)